# ⚡ Notebook 2 — Rapid Prototyping (AWS Bedrock)

Go from 'it works' to 'it's reliable, fast, and measurable.'

1. Sequential → Async (speed)
2. No retry → Exponential backoff (resilience)
3. No cost tracking → Cost calculator (awareness)
4. Prompt v1 → Prompt v2 (quality)
5. Full production pipeline

---

In [2]:
import boto3, json, time, asyncio
import pandas as pd
from IPython.display import display
from concurrent.futures import ThreadPoolExecutor

REGION  = 'us-east-1'
MODEL   = 'us.anthropic.claude-sonnet-4-6'
PRICE   = {'input': 3.00, 'output': 15.00}  # per 1M tokens

session = boto3.Session(profile_name='default', region_name=REGION)
client  = session.client('bedrock-runtime')

def call_bedrock(prompt, max_tokens=80, temperature=0.7, system=None):
    body = {'anthropic_version':'bedrock-2023-05-31','max_tokens':max_tokens,'temperature':temperature,'messages':[{'role':'user','content':prompt}]}
    if system:
        body['system'] = system
    start  = time.time()
    resp   = client.invoke_model(modelId=MODEL, body=json.dumps(body))
    result = json.loads(resp['body'].read())
    return result['content'][0]['text'], round((time.time()-start)*1000), result['usage']

def calculate_cost(usage):
    return round((usage['input_tokens']*PRICE['input'] + usage['output_tokens']*PRICE['output'])/1_000_000, 6)

print('✅ Setup complete')

✅ Setup complete


---
## Section 1 — Sequential vs Async
> Five calls. Sequential waits for each. Async fires all at once. Watch the timing.

In [3]:
PROMPTS = [
    'Name one famous scientist in exactly 5 words.',
    'Name one famous painter in exactly 5 words.',
    'Name one famous musician in exactly 5 words.',
    'Name one famous mathematician in exactly 5 words.',
    'Name one famous engineer in exactly 5 words.',
]

# ❌ SEQUENTIAL — each call blocks until done
print('❌ Sequential calls:\n')
seq_start = time.time()
for i, p in enumerate(PROMPTS):
    t0   = time.time()
    text, latency, _ = call_bedrock(p, max_tokens=20)
    print(f'  Call {i+1}: {latency}ms — {text.strip()}')
total_seq = round(time.time() - seq_start, 2)
print(f'\n⏱ Total (sequential): {total_seq}s')

❌ Sequential calls:

  Call 1: 3619ms — **Albert Einstein, theory of relativity.**
  Call 2: 1578ms — Leonardo da Vinci painted masterpieces.
  Call 3: 1545ms — **Elvis Presley, the King of Rock.**

Wait, that's more than 5
  Call 4: 1564ms — **Leonhard Euler, Swiss math genius.**
  Call 5: 1976ms — **Nikola Tesla, electrical engineering pioneer.**

⏱ Total (sequential): 10.29s


In [4]:
# ✅ ASYNC — all five calls fire simultaneously using ThreadPoolExecutor
# (boto3 is synchronous; we wrap it in asyncio via run_in_executor)
print('✅ Async calls (concurrent):\n')
_executor = ThreadPoolExecutor(max_workers=5)

async def async_call(prompt, idx):
    loop = asyncio.get_event_loop()
    t0   = time.time()
    text, _, _ = await loop.run_in_executor(_executor, lambda: call_bedrock(prompt, max_tokens=20))
    return idx, round(time.time()-t0, 2), text.strip()

async def run_all():
    tasks = [async_call(p, i+1) for i, p in enumerate(PROMPTS)]
    return await asyncio.gather(*tasks)

async_start = time.time()
results = await run_all()
total_async = round(time.time() - async_start, 2)

for idx, elapsed, text in sorted(results):
    print(f'  Call {idx}: {elapsed}s — {text}')

print(f'\n⏱ Total (async): {total_async}s')
print(f'\n🚀 Speedup: {total_seq}s → {total_async}s  ({total_seq/total_async:.1f}x faster)')
print('\n👉 The more calls you make, the bigger the async advantage.')

✅ Async calls (concurrent):

  Call 1: 1.55s — **Albert Einstein, theory of relativity.**
  Call 2: 4.92s — Leonardo da Vinci painted masterpieces.
  Call 3: 5.03s — **Freddie Mercury, Queen's iconic singer.**
  Call 4: 5.23s — **Leonhard Euler, Swiss math genius.**
  Call 5: 5.63s — **Nikola Tesla, electrical engineering pioneer.**

⏱ Total (async): 5.67s

🚀 Speedup: 10.29s → 5.67s  (1.8x faster)

👉 The more calls you make, the bigger the async advantage.


---
## Section 2 — Error Handling & Retry
> APIs fail. Show the crash, then the graceful recovery.

In [5]:
# Simulate a call that fails the first 2 times (mimics 429)
call_counter = [0]

def unreliable_call(prompt, fail_times=2):
    call_counter[0] += 1
    if call_counter[0] <= fail_times:
        raise Exception(f'429 ThrottlingException — simulated failure #{call_counter[0]}')
    return call_bedrock(prompt)

# ❌ NO RETRY — crashes immediately
print('❌ Without retry:\n')
try:
    result = unreliable_call('What is AI?')
    print(result[0])
except Exception as e:
    print(f'💥 App crashed: {e}')
    print('\n👉 User sees an error. Session is lost.')

❌ Without retry:

💥 App crashed: 429 ThrottlingException — simulated failure #1

👉 User sees an error. Session is lost.


In [6]:
# ✅ WITH RETRY — exponential backoff
call_counter[0] = 0  # reset

def call_with_retry(prompt, max_retries=4):
    for attempt in range(1, max_retries + 1):
        try:
            return unreliable_call(prompt)
        except Exception as e:
            wait = 2 ** (attempt - 1)  # 1s → 2s → 4s → 8s
            if attempt == max_retries:
                print(f'  ❌ All {max_retries} attempts exhausted.')
                raise
            print(f'  Attempt {attempt} failed: {e}')
            print(f'  Waiting {wait}s before retry...')
            time.sleep(wait)

print('✅ With exponential backoff:\n')
text, latency, _ = call_with_retry('What is AI?')
print(f'\n✅ Final result: {text.strip()[:120]}')
print('\n👉 User never sees the error. App recovered silently.')

✅ With exponential backoff:

  Attempt 1 failed: 429 ThrottlingException — simulated failure #1
  Waiting 1s before retry...
  Attempt 2 failed: 429 ThrottlingException — simulated failure #2
  Waiting 2s before retry...

✅ Final result: # What is AI?

**Artificial Intelligence (AI)** refers to the development of computer systems that can perform tasks tha

👉 User never sees the error. App recovered silently.


---
## Section 3 — Cost Calculator
> Without tracking you won't know you're overspending until the bill arrives.

In [7]:
ml_prompts = [
    'Summarize machine learning in 2 sentences.',
    'What is overfitting?',
    'Explain gradient descent simply.',
]

# ❌ NO COST TRACKING
print('❌ Calling model with no cost tracking:\n')
for p in ml_prompts:
    text, _, _ = call_bedrock(p)
    print(f'  {p[:40]}...')

print('\n❌ How much did that cost? Which prompt was most expensive? → Cannot answer.')

❌ Calling model with no cost tracking:

  Summarize machine learning in 2 sentence...
  What is overfitting?...
  Explain gradient descent simply....

❌ How much did that cost? Which prompt was most expensive? → Cannot answer.


In [8]:
# ✅ WITH COST TRACKING
cost_log = []

for p in ml_prompts:
    text, latency, usage = call_bedrock(p)
    cost = calculate_cost(usage)
    cost_log.append({
        'prompt':        p[:38] + '...',
        'input_tokens':  usage['input_tokens'],
        'output_tokens': usage['output_tokens'],
        'latency_ms':    latency,
        'cost_usd':      cost,
    })

df = pd.DataFrame(cost_log)
display(df)
print(f'\nTotal cost: ${df["cost_usd"].sum():.6f}')
print(f'Avg latency: {df["latency_ms"].mean():.0f}ms')
print('\n👉 Now you can answer both questions.')

,prompt,input_tokens,output_tokens,latency_ms,cost_usd
0,Summarize machine learning in 2 senten...,18,58,2389,0.000924
1,What is overfitting?...,13,80,2881,0.001239
2,Explain gradient descent simply....,13,79,2412,0.001224



Total cost: $0.003387
Avg latency: 2561ms

👉 Now you can answer both questions.


---
## Section 4 — Prompt Versioning
> Prompts are code. Version them, track which version ran, A/B compare.

In [9]:
def load_prompt(version, **kwargs):
    with open(f'../app/prompts/{version}.txt') as f:
        return f.read().format(**kwargs)

ARTICLE = """
Transformers are a neural network architecture that uses self-attention to process
sequential data. Unlike RNNs, they handle all tokens in parallel, making training
much faster. They form the backbone of modern LLMs such as GPT and Claude.
"""
print('Helper ready.')

Helper ready.


In [10]:
# ❌ v1 — vague instruction
prompt_v1 = load_prompt('v1', text=ARTICLE)
print(f'❌ v1 template: {repr(prompt_v1[:60])}')
print()
text_v1, _, usage_v1 = call_bedrock(prompt_v1, max_tokens=200)
print('Output:')
print(text_v1)
print(f'\nOutput tokens: {usage_v1["output_tokens"]}')

❌ v1 template: 'Summarize this text: \nTransformers are a neural network arch'

Output:
## Summary

Transformers are a neural network architecture that processes sequential data using self-attention. Unlike RNNs, they handle all tokens simultaneously (in parallel), enabling faster training. They serve as the foundation for modern large language models like GPT and Claude.

Output tokens: 58


In [11]:
# ✅ v2 — specific, structured instruction
prompt_v2 = load_prompt('v2', text=ARTICLE)
print(f'✅ v2 template: {repr(prompt_v2[:100])}')
print()
text_v2, _, usage_v2 = call_bedrock(prompt_v2, max_tokens=200)
print('Output:')
print(text_v2)
print(f'\nOutput tokens: {usage_v2["output_tokens"]}')

comp = pd.DataFrame([
    {'version':'v1','template':'Summarize this text: {text}','output_tokens':usage_v1['output_tokens'],'structured':'❌'},
    {'version':'v2','template':'3 bullets, action verbs, 1 sentence each','output_tokens':usage_v2['output_tokens'],'structured':'✅'},
])
display(comp)
print('\n👉 v2 gives structured output — predictable, parseable, easier to use downstream.')

✅ v2 template: 'Summarize the following text in exactly 3 bullet points.\n- Each bullet must start with a strong acti'

Output:
• **Utilize** self-attention mechanisms to process sequential data in parallel, unlike traditional RNNs.
• **Enable** significantly faster training by handling all tokens simultaneously rather than sequentially.
• **Power** modern large language models such as GPT and Claude as their foundational architecture.

Output tokens: 65


,version,template,output_tokens,structured
0,v1,Summarize this text: {text},58,❌
1,v2,"3 bullets, action verbs, 1 sentence each",65,✅



👉 v2 gives structured output — predictable, parseable, easier to use downstream.


---
## Section 5 — Full Production Pipeline
> Combine: validation → versioned prompt → retry → cost tracking → output validation → log.

In [12]:
pipeline_log = []

def production_call(text, prompt_version='v2'):
    # 1. Input validation
    if not text or len(text.strip()) < 10:
        raise ValueError('Input too short (min 10 characters)')
    if len(text) > 10_000:
        raise ValueError('Input exceeds 10,000 character limit')

    # 2. Load versioned prompt
    try:
        prompt = load_prompt(prompt_version, text=text)
    except (FileNotFoundError, KeyError):
        prompt = f'Summarize this text: {text}'  # safe fallback

    # 3. Call with retry
    output, latency, usage = None, 0, {}
    for attempt in range(1, 4):
        try:
            output, latency, usage = call_bedrock(prompt, max_tokens=200)
            break
        except Exception:
            if attempt == 3:
                raise
            time.sleep(2 ** (attempt - 1))

    # 4. Output validation
    if not output or not output.strip():
        raise ValueError('Model returned empty response')

    # 5. Log
    pipeline_log.append({
        'prompt_version': prompt_version,
        'input_tokens':   usage['input_tokens'],
        'output_tokens':  usage['output_tokens'],
        'latency_ms':     latency,
        'cost_usd':       calculate_cost(usage),
        'status':         'success',
    })
    return output.strip()

result = production_call(ARTICLE)
print('✅ Pipeline result:\n')
print(result)
print('\nPipeline log:')
display(pd.DataFrame(pipeline_log))

✅ Pipeline result:

• **Utilize** self-attention mechanisms to process sequential data in parallel, unlike traditional RNNs.
• **Enable** significantly faster training by handling all tokens simultaneously rather than sequentially.
• **Power** modern large language models such as GPT and Claude as their core architectural foundation.

Pipeline log:


,prompt_version,input_tokens,output_tokens,latency_ms,cost_usd,status
0,v2,111,65,2293,0.001308,success


In [13]:
# Validation catching bad inputs
test_cases = [('', 'empty'), ('hi', 'too short'), ('x'*11_000, 'too long')]

print('Validation test cases:\n')
for bad_input, label in test_cases:
    try:
        production_call(bad_input)
        print(f'  [{label}] Passed (unexpected!)')
    except ValueError as e:
        print(f'  [{label}] ✅ Caught: {e}')

Validation test cases:

  [empty] ✅ Caught: Input too short (min 10 characters)
  [too short] ✅ Caught: Input too short (min 10 characters)
  [too long] ✅ Caught: Input exceeds 10,000 character limit


---
## ✅ Notebook 2 Summary

| Concept | Without | With |
|---|---|---|
| Async | ~12s for 5 calls | ~3s (4x faster) |
| Retry | App crashes on 429 | Recovers silently |
| Cost tracking | Blind to spend | Per-call cost table |
| Prompt versioning | Vague output | Structured, parseable |
| Full pipeline | None | Validate → Call → Log → Return |

**Next:** Run the Streamlit app:
```bash
cd app && streamlit run app.py
```